# README Qualitative Visualizations

This notebook generates README-friendly qualitative examples for **ACDC**, **M\&M**, and **STONE**.

## What predictions are used

The visualizations are generated from baseline checkpoints and fresh slice-wise inference:

- **ACDC**: `exp67_uniform_l1_acdc/baseline/training/final_model.pth`
- **M\&M**: `exp74_uniform_l1_MM/baseline/training/final_model.pth`
- **STONE**: `exp54_stone/baseline/training/final_model.pth`

For STONE, the newer `exp85_stone_l1` baseline metadata appears to contain copied ACDC paths, so this notebook uses the older `exp54_stone` baseline config/checkpoint explicitly.

## Selected cases and outputs

- One representative case is selected automatically per dataset unless manually overridden below.
- The selected cases are printed in the **Selection summary** table near the top of the notebook.
- Outputs are written to `basic_UNet/results/qualitative_readme/<dataset>/`.

Each dataset export contains:

- one GIF scrolling through the slices in stored anatomical order
- one folder with per-slice `4`-panel frames
- one static middle-slice PNG
- one static first/middle/last overlay summary PNG


In [1]:
from __future__ import annotations

import math
import re
import warnings
from functools import lru_cache
from pathlib import Path
import sys

REPO_ROOT = Path('/mnt/hdd/ttoxopeus/basic_UNet')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
import yaml
from IPython.display import display
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap
from PIL import Image

from src.models.unet import UNet
OUTPUT_ROOT = REPO_ROOT / 'docs' / 'readme_assets'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


## Configuration

The dataset-specific configuration below keeps the notebook explicit and robust.

- `kind="acdcmm"` expects ACDC/M\&M naming such as `patient101_ED.nii.gz`
- `kind="stone"` expects STONE naming such as `0001_2_0000.nii.gz`
- `MANUAL_CASE_SELECTION` can be set to exact case IDs if you want fixed examples


In [2]:
CLASS_INFO = [
    ('RV', 1, '#F58518'),
    ('MYO', 2, '#54A24B'),
    ('LV', 3, '#4C78A8'),
]

CLASS_CMAP = ListedColormap(['#000000'] + [color for _, _, color in CLASS_INFO])

DATASET_CONFIGS = {
    'acdc': {
        'display_name': 'ACDC',
        'kind': 'acdcmm',
        'config_path': REPO_ROOT / 'results/UNet_ACDC/exp67_uniform_l1_acdc/baseline/config.yaml',
        'checkpoint_path': REPO_ROOT / 'results/UNet_ACDC/exp67_uniform_l1_acdc/baseline/training/final_model.pth',
        'images_ts': Path('/mnt/hdd/ttoxopeus/datasets/nnUNet_raw/Dataset200_ACDC/imagesTs'),
        'labels_ts': Path('/mnt/hdd/ttoxopeus/datasets/nnUNet_raw/Dataset200_ACDC/labelsTs'),
        'slice_axis': 2,
    },
    'mm': {
        'display_name': 'M&M',
        'kind': 'acdcmm',
        'config_path': REPO_ROOT / 'results/UNet_ACDC/exp74_uniform_l1_MM/baseline/config.yaml',
        'checkpoint_path': REPO_ROOT / 'results/UNet_ACDC/exp74_uniform_l1_MM/baseline/training/final_model.pth',
        'images_ts': Path('/mnt/hdd/ttoxopeus/datasets/nnUNet_raw/Dataset300_MM/imagesTs'),
        'labels_ts': Path('/mnt/hdd/ttoxopeus/datasets/nnUNet_raw/Dataset300_MM/labelsTs'),
        'slice_axis': 2,
    },
    'stone': {
        'display_name': 'STONE',
        'kind': 'stone',
        'config_path': REPO_ROOT / 'results/UNet_ACDC/exp54_stone/baseline/config.yaml',
        'checkpoint_path': REPO_ROOT / 'results/UNet_ACDC/exp54_stone/baseline/training/final_model.pth',
        'images_ts': Path('/mnt/hdd/ttoxopeus/datasets/nnUNet_raw/Dataset100_STONE/imagesTs'),
        'labels_ts': Path('/mnt/hdd/ttoxopeus/datasets/nnUNet_raw/Dataset100_STONE/labelsTs'),
        'slice_axis': 2,
    },
}

MANUAL_CASE_SELECTION = {
    'acdc': None,
    'mm': None,
    'stone': None,
}

GIF_DURATION_MS = 450
FRAME_DPI = 140
TARGET_SIZE = (256, 256)
MIDDLE_SLICE_KEY = 'middle_slice'

for dataset_key, cfg in DATASET_CONFIGS.items():
    if not cfg['images_ts'].exists():
        warnings.warn(f"Missing image directory for {dataset_key}: {cfg['images_ts']}")
    if not cfg['labels_ts'].exists():
        warnings.warn(f"Missing label directory for {dataset_key}: {cfg['labels_ts']}")
    if not cfg['checkpoint_path'].exists():
        warnings.warn(f"Missing checkpoint for {dataset_key}: {cfg['checkpoint_path']}")


## Reused utilities and assumptions

This notebook reuses the same core ideas as `baseline_analysis.ipynb`:

- percentile clipping and min-max normalization before resizing
- per-slice inference from stored NIfTI volumes
- the class colors `RV=#F58518`, `MYO=#54A24B`, `LV=#4C78A8`
- grayscale input with contour-based overlay visualization

Slice ordering is taken from the stored volume order along `axis=2`. No explicit affine-based verification of basal-to-apical orientation was found elsewhere in the repo, so this notebook documents and uses the stored order directly.


In [3]:
def unwrap_state_dict(obj):
    if isinstance(obj, dict):
        for key in ['state_dict', 'model_state_dict', 'net', 'model']:
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def hex_to_rgb01(color: str):
    return tuple(int(color[i:i+2], 16) / 255.0 for i in (1, 3, 5))




def orient_for_display(dataset_key: str, arr: np.ndarray) -> np.ndarray:
    if dataset_key in {'acdc', 'mm'}:
        return np.flipud(np.rot90(arr, 1))
    return arr


def build_mask_rgba(mask: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    rgba = np.zeros((*mask.shape, 4), dtype=float)
    for _, class_idx, color in CLASS_INFO:
        rgb = hex_to_rgb01(color)
        class_mask = mask == class_idx
        if not np.any(class_mask):
            continue
        rgba[class_mask, 0] = rgb[0]
        rgba[class_mask, 1] = rgb[1]
        rgba[class_mask, 2] = rgb[2]
        rgba[class_mask, 3] = alpha
    return rgba


def extract_case_metadata(dataset_key: str, label_path: Path) -> dict:
    stem = label_path.name.replace('.nii.gz', '')
    if dataset_key in {'acdc', 'mm'}:
        image_name = label_path.name.replace('.nii.gz', '_0000.nii.gz')
        image_path = DATASET_CONFIGS[dataset_key]['images_ts'] / image_name
        match = re.match(r'(?P<patient>.+?)_(?P<phase>ED|ES)$', stem)
        patient_id = match.group('patient') if match else stem
        phase = match.group('phase') if match else 'Unknown'
        return {
            'case_id': stem,
            'patient_id': patient_id,
            'phase': phase,
            'display_id': f'{patient_id} {phase}',
            'image_path': image_path,
            'label_path': label_path,
        }

    image_path = DATASET_CONFIGS[dataset_key]['images_ts'] / label_path.name
    tokens = stem.split('_')
    patient_id = tokens[0]
    sequence_id = tokens[1] if len(tokens) > 1 else '0'
    return {
        'case_id': stem,
        'patient_id': patient_id,
        'phase': sequence_id,
        'display_id': f'{patient_id} seq {sequence_id}',
        'image_path': image_path,
        'label_path': label_path,
    }


def collect_cases(dataset_key: str) -> list[dict]:
    cfg = DATASET_CONFIGS[dataset_key]
    cases = []
    for label_path in sorted(cfg['labels_ts'].glob('*.nii.gz')):
        meta = extract_case_metadata(dataset_key, label_path)
        if not meta['image_path'].exists():
            warnings.warn(f"Missing image for label {label_path.name}: expected {meta['image_path']}")
            continue
        cases.append(meta)
    return cases


def preprocess_image_slice(img_np: np.ndarray, target_size=TARGET_SIZE, clip_bounds=None):
    if clip_bounds is None:
        lo, hi = np.percentile(img_np, 1), np.percentile(img_np, 99)
    else:
        lo, hi = clip_bounds
    img_np = np.clip(img_np, lo, hi)
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    img_t = torch.from_numpy(img_np).float().unsqueeze(0)
    img_t = TF.resize(img_t, target_size, antialias=True)
    return img_t


def preprocess_label_slice(lbl_np: np.ndarray, target_size=TARGET_SIZE):
    lbl_t = torch.from_numpy(lbl_np).long().unsqueeze(0).float()
    lbl_t = TF.resize(lbl_t, target_size, interpolation=TF.InterpolationMode.NEAREST)
    return lbl_t.squeeze(0).long()


def dice_binary(pred_mask: np.ndarray, gt_mask: np.ndarray, eps: float = 1e-6) -> float:
    inter = np.logical_and(pred_mask, gt_mask).sum()
    union = pred_mask.sum() + gt_mask.sum()
    return float((2.0 * inter + eps) / (union + eps))


def dice_per_class(pred_lbl: np.ndarray, gt_lbl: np.ndarray, num_classes: int = 4) -> dict[int, float]:
    return {c: dice_binary(pred_lbl == c, gt_lbl == c) for c in range(num_classes)}


@lru_cache(maxsize=None)
def load_model(dataset_key: str):
    cfg = DATASET_CONFIGS[dataset_key]
    model_cfg = yaml.safe_load(cfg['config_path'].read_text())['train']['model']
    model = UNet(
        in_ch=int(model_cfg['in_channels']),
        out_ch=int(model_cfg['out_channels']),
        enc_features=tuple(model_cfg['features']),
    ).to(DEVICE)
    state = torch.load(cfg['checkpoint_path'], map_location=DEVICE)
    model.load_state_dict(unwrap_state_dict(state))
    model.eval()
    return model


CASE_CACHE: dict[tuple[str, str], dict] = {}


@torch.no_grad()
def run_case_inference(dataset_key: str, case_meta: dict) -> dict:
    cache_key = (dataset_key, case_meta['case_id'])
    if cache_key in CASE_CACHE:
        return CASE_CACHE[cache_key]

    cfg = DATASET_CONFIGS[dataset_key]
    model = load_model(dataset_key)

    img_vol = nib.load(str(case_meta['image_path'])).get_fdata()
    lbl_vol = nib.load(str(case_meta['label_path'])).get_fdata().astype(np.int64)
    axis = int(cfg['slice_axis'])
    n_slices = int(lbl_vol.shape[axis])
    clip_bounds = tuple(np.percentile(img_vol, [1, 99]).tolist())

    slice_records = []
    for slice_idx in range(n_slices):
        img_slice = np.take(img_vol, slice_idx, axis=axis)
        gt_raw = np.take(lbl_vol, slice_idx, axis=axis).astype(np.int64)

        x = preprocess_image_slice(img_slice, clip_bounds=clip_bounds)
        y = preprocess_label_slice(gt_raw)
        pred = torch.argmax(model(x.unsqueeze(0).to(DEVICE)), dim=1).squeeze(0).cpu().numpy()
        gt = y.cpu().numpy()
        img_norm = x.squeeze(0).cpu().numpy()
        class_dice = dice_per_class(pred, gt, num_classes=4)
        fg_dice = float(np.mean([class_dice[c] for c in (1, 2, 3)]))

        slice_records.append({
            'slice_idx': slice_idx,
            'img': img_norm,
            'gt': gt,
            'pred': pred,
            'fg_dice': fg_dice,
            'class_dice': class_dice,
            'gt_fg_pixels': int((gt > 0).sum()),
        })

    payload = {
        'dataset_key': dataset_key,
        'case_meta': case_meta,
        'clip_bounds': clip_bounds,
        'num_slices': n_slices,
        'slice_records': slice_records,
        'mean_fg_dice': float(np.mean([row['fg_dice'] for row in slice_records])),
        'fg_slice_fraction': float(np.mean([row['gt_fg_pixels'] > 0 for row in slice_records])),
        'mean_fg_pixels': float(np.mean([row['gt_fg_pixels'] for row in slice_records])),
    }
    CASE_CACHE[cache_key] = payload
    return payload


## Selection summary

To avoid choosing a trivial or pathological failure case, the notebook scores every case using a lightweight representative subset of foreground slices and then picks a case near the dataset median. Manual overrides remain possible through `MANUAL_CASE_SELECTION`.


In [4]:
def representative_slice_indices(label_volume: np.ndarray, axis: int) -> list[int]:
    n_slices = int(label_volume.shape[axis])
    fg_indices = [idx for idx in range(n_slices) if np.take(label_volume, idx, axis=axis).astype(np.int64).any()]
    if not fg_indices:
        return [n_slices // 2]
    anchors = np.linspace(0, len(fg_indices) - 1, num=min(3, len(fg_indices))).round().astype(int)
    return sorted({int(fg_indices[idx]) for idx in anchors})


@torch.no_grad()
def quick_case_score(dataset_key: str, case_meta: dict) -> dict:
    cfg = DATASET_CONFIGS[dataset_key]
    model = load_model(dataset_key)
    img_vol = nib.load(str(case_meta['image_path'])).get_fdata()
    lbl_vol = nib.load(str(case_meta['label_path'])).get_fdata().astype(np.int64)
    axis = int(cfg['slice_axis'])
    clip_bounds = tuple(np.percentile(img_vol, [1, 99]).tolist())
    sample_indices = representative_slice_indices(lbl_vol, axis)

    values = []
    fg_pixels = []
    for slice_idx in sample_indices:
        img_slice = np.take(img_vol, slice_idx, axis=axis)
        gt_raw = np.take(lbl_vol, slice_idx, axis=axis).astype(np.int64)
        x = preprocess_image_slice(img_slice, clip_bounds=clip_bounds)
        y = preprocess_label_slice(gt_raw)
        pred = torch.argmax(model(x.unsqueeze(0).to(DEVICE)), dim=1).squeeze(0).cpu().numpy()
        gt = y.cpu().numpy()
        values.append(float(np.mean([dice_binary(pred == c, gt == c) for c in (1, 2, 3)])))
        fg_pixels.append(int((gt > 0).sum()))

    n_slices = int(lbl_vol.shape[axis])
    fg_slice_fraction = float(np.mean([(np.take(lbl_vol, idx, axis=axis) > 0).any() for idx in range(n_slices)]))
    return {
        'dataset': dataset_key,
        'case_id': case_meta['case_id'],
        'patient_id': case_meta['patient_id'],
        'phase': case_meta['phase'],
        'display_id': case_meta['display_id'],
        'n_slices': n_slices,
        'quick_fg_dice': float(np.mean(values)),
        'fg_slice_fraction': fg_slice_fraction,
        'sampled_fg_pixels': float(np.mean(fg_pixels)),
        'sample_indices': sample_indices,
    }


def build_selection_table(dataset_key: str) -> pd.DataFrame:
    rows = []
    for case_meta in collect_cases(dataset_key):
        try:
            rows.append(quick_case_score(dataset_key, case_meta))
        except Exception as exc:
            warnings.warn(f"Skipping {dataset_key}:{case_meta['case_id']} because scoring failed: {exc}")
    if not rows:
        raise RuntimeError(f'No scorable cases found for {dataset_key}')
    return pd.DataFrame(rows)


def auto_select_case(selection_df: pd.DataFrame) -> pd.Series:
    q_lo = selection_df['quick_fg_dice'].quantile(0.35)
    q_hi = selection_df['quick_fg_dice'].quantile(0.70)
    fg_med = selection_df['sampled_fg_pixels'].median()
    n_med = selection_df['n_slices'].median()
    eligible = selection_df[
        selection_df['quick_fg_dice'].between(q_lo, q_hi)
        & (selection_df['fg_slice_fraction'] >= 0.45)
        & (selection_df['sampled_fg_pixels'] > 0)
    ].copy()
    if eligible.empty:
        eligible = selection_df.copy()
    eligible['selection_score'] = (
        (eligible['quick_fg_dice'] - selection_df['quick_fg_dice'].median()).abs()
        + 0.03 * (eligible['n_slices'] - n_med).abs()
        + 0.00002 * (eligible['sampled_fg_pixels'] - fg_med).abs()
    )
    return eligible.sort_values(['selection_score', 'quick_fg_dice'], ascending=[True, False]).iloc[0]


selection_tables = {dataset_key: build_selection_table(dataset_key) for dataset_key in DATASET_CONFIGS}
selected_cases = {}
for dataset_key, selection_df in selection_tables.items():
    manual_case = MANUAL_CASE_SELECTION.get(dataset_key)
    if manual_case is not None:
        match = selection_df[selection_df['case_id'] == manual_case]
        if match.empty:
            warnings.warn(f"Manual case {manual_case!r} was not found for {dataset_key}; falling back to automatic selection.")
            chosen = auto_select_case(selection_df)
        else:
            chosen = match.iloc[0]
    else:
        chosen = auto_select_case(selection_df)
    selected_cases[dataset_key] = chosen.to_dict()

selection_summary = pd.DataFrame([
    {
        'dataset': DATASET_CONFIGS[key]['display_name'],
        'selected_case': row['case_id'],
        'display_id': row['display_id'],
        'quick_fg_dice': round(float(row['quick_fg_dice']), 4),
        'n_slices': int(row['n_slices']),
        'fg_slice_fraction': round(float(row['fg_slice_fraction']), 3),
        'output_dir': str(OUTPUT_ROOT / key),
    }
    for key, row in selected_cases.items()
])

display(selection_summary)


,dataset,selected_case,display_id,quick_fg_dice,n_slices,fg_slice_fraction,output_dir
0,ACDC,patient145_ED,patient145 ED,0.8333,10,0.900,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...
1,M&M,patient105_ES,patient105 ES,0.7862,12,0.667,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...
2,STONE,0185_2_0000,0185 seq 2,0.5245,11,1.000,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...


## Rendering and export

Each slice is rendered as a fixed 4-panel figure with the exact layout:

`[ Input | GT | Prediction | Overlay ]`

The overlay uses:

- grayscale MRI input
- GT semi-transparent fills plus solid contours
- prediction dashed contours
- red mismatch highlighting for disagreement regions

All panels are saved to disk, then combined into a GIF for each dataset.


In [5]:
def render_slice_figure(dataset_key: str, case_payload: dict, slice_record: dict):
    display_name = DATASET_CONFIGS[dataset_key]['display_name']
    case_meta = case_payload['case_meta']
    slice_num = slice_record['slice_idx'] + 1
    n_slices = case_payload['num_slices']

    fig, axes = plt.subplots(1, 4, figsize=(13.6, 3.8), dpi=FRAME_DPI)
    fig.patch.set_facecolor('white')

    img = orient_for_display(dataset_key, slice_record['img'])
    gt = orient_for_display(dataset_key, slice_record['gt'])
    pred = orient_for_display(dataset_key, slice_record['pred'])

    axes[0].imshow(img, cmap='gray', vmin=0.0, vmax=1.0)
    axes[0].set_title('Input', fontsize=12, fontweight='bold')

    axes[1].imshow(gt, cmap=CLASS_CMAP, vmin=0, vmax=3)
    axes[1].set_title('GT', fontsize=12, fontweight='bold')

    axes[2].imshow(pred, cmap=CLASS_CMAP, vmin=0, vmax=3)
    axes[2].set_title('Prediction', fontsize=12, fontweight='bold')

    axes[3].imshow(img, cmap='gray', vmin=0.0, vmax=1.0)
    gt_rgba = build_mask_rgba(gt, alpha=0.22)
    if gt_rgba[..., 3].any():
        axes[3].imshow(gt_rgba)

    mismatch_mask = ((gt > 0) | (pred > 0)) & (gt != pred)
    if mismatch_mask.any():
        mismatch_rgba = np.zeros((*mismatch_mask.shape, 4), dtype=float)
        mismatch_rgba[..., 0] = 1.0
        mismatch_rgba[..., 3] = mismatch_mask.astype(float) * 0.30
        axes[3].imshow(mismatch_rgba)

    for _, class_idx, color in CLASS_INFO:
        gt_mask = (gt == class_idx).astype(float)
        pred_mask = (pred == class_idx).astype(float)
        if gt_mask.any():
            axes[3].contour(gt_mask, levels=[0.5], colors=['white'], linewidths=1.9)
            axes[3].contour(gt_mask, levels=[0.5], colors=[color], linewidths=1.1)
        if pred_mask.any():
            axes[3].contour(pred_mask, levels=[0.5], colors=['black'], linewidths=1.8, linestyles='--')
            axes[3].contour(pred_mask, levels=[0.5], colors=[color], linewidths=1.0, linestyles='--')
    axes[3].set_title('Overlay', fontsize=12, fontweight='bold')

    for ax in axes:
        ax.axis('off')

    legend_handles = [
        Line2D([0], [0], color=color, lw=4, label=name)
        for name, _, color in CLASS_INFO
    ]
    legend_handles += [
        Line2D([0], [0], color='black', lw=2.2, linestyle='-', label='GT contour'),
        Line2D([0], [0], color='black', lw=2.2, linestyle='--', label='Prediction contour'),
    ]

    fig.suptitle(
        f"{display_name} | {case_meta['display_id']} | Slice {slice_num}/{n_slices} | FG Dice={slice_record['fg_dice']:.3f}",
        fontsize=13,
        fontweight='bold',
        y=0.98,
    )
    fig.legend(handles=legend_handles, loc='lower center', ncol=5, frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.01))
    plt.tight_layout(rect=(0, 0.08, 1, 0.93))
    return fig


def save_middle_slice_png(dataset_key: str, case_payload: dict, dataset_out: Path):
    middle_slice = case_payload['slice_records'][len(case_payload['slice_records']) // 2]
    fig = render_slice_figure(dataset_key, case_payload, middle_slice)
    out_path = dataset_out / 'middle_slice_4panel.png'
    fig.savefig(out_path, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return out_path


def save_overlay_summary_png(dataset_key: str, case_payload: dict, dataset_out: Path):
    slice_records = case_payload['slice_records']
    key_indices = [0, len(slice_records) // 2, len(slice_records) - 1]
    key_indices = list(dict.fromkeys(key_indices))

    fig, axes = plt.subplots(1, len(key_indices), figsize=(4.4 * len(key_indices), 4.2), dpi=FRAME_DPI)
    if len(key_indices) == 1:
        axes = [axes]

    for ax, idx in zip(axes, key_indices):
        row = slice_records[idx]
        img = orient_for_display(dataset_key, row['img'])
        gt = orient_for_display(dataset_key, row['gt'])
        pred = orient_for_display(dataset_key, row['pred'])
        ax.imshow(img, cmap='gray', vmin=0.0, vmax=1.0)
        gt_rgba = build_mask_rgba(gt, alpha=0.22)
        if gt_rgba[..., 3].any():
            ax.imshow(gt_rgba)
        mismatch_mask = ((gt > 0) | (pred > 0)) & (gt != pred)
        if mismatch_mask.any():
            mismatch_rgba = np.zeros((*mismatch_mask.shape, 4), dtype=float)
            mismatch_rgba[..., 0] = 1.0
            mismatch_rgba[..., 3] = mismatch_mask.astype(float) * 0.30
            ax.imshow(mismatch_rgba)
        for _, class_idx, color in CLASS_INFO:
            gt_mask = (gt == class_idx).astype(float)
            pred_mask = (pred == class_idx).astype(float)
            if gt_mask.any():
                ax.contour(gt_mask, levels=[0.5], colors=['white'], linewidths=1.7)
                ax.contour(gt_mask, levels=[0.5], colors=[color], linewidths=1.0)
            if pred_mask.any():
                ax.contour(pred_mask, levels=[0.5], colors=['black'], linewidths=1.6, linestyles='--')
                ax.contour(pred_mask, levels=[0.5], colors=[color], linewidths=0.9, linestyles='--')
        ax.set_title(f"Slice {row['slice_idx'] + 1}/{case_payload['num_slices']}\nFG Dice={row['fg_dice']:.3f}", fontsize=11, fontweight='bold')
        ax.axis('off')

    fig.suptitle(f"{DATASET_CONFIGS[dataset_key]['display_name']} | {case_payload['case_meta']['display_id']} | Overlay summary", fontsize=13, fontweight='bold')
    plt.tight_layout(rect=(0, 0, 1, 0.92))
    out_path = dataset_out / 'overlay_summary.png'
    fig.savefig(out_path, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return out_path


def export_dataset_visuals(dataset_key: str, selected_row: dict) -> dict:
    dataset_out = OUTPUT_ROOT
    frames_dir = OUTPUT_ROOT / f'{dataset_key}_frames'
    frames_dir.mkdir(parents=True, exist_ok=True)

    case_meta = next(meta for meta in collect_cases(dataset_key) if meta['case_id'] == selected_row['case_id'])
    case_payload = run_case_inference(dataset_key, case_meta)

    frame_paths = []
    for row in case_payload['slice_records']:
        fig = render_slice_figure(dataset_key, case_payload, row)
        frame_path = frames_dir / f"slice_{row['slice_idx']:03d}.png"
        fig.savefig(frame_path, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        frame_paths.append(frame_path)

    gif_path = OUTPUT_ROOT / f"{dataset_key}_qualitative.gif"
    images = [Image.open(frame).convert('P', palette=Image.ADAPTIVE) for frame in frame_paths]
    if images:
        images[0].save(
            gif_path,
            save_all=True,
            append_images=images[1:],
            duration=GIF_DURATION_MS,
            loop=0,
            optimize=False,
            disposal=2,
        )

    middle_png = save_middle_slice_png(dataset_key, case_payload, dataset_out)
    summary_png = save_overlay_summary_png(dataset_key, case_payload, dataset_out)

    return {
        'dataset': DATASET_CONFIGS[dataset_key]['display_name'],
        'case_id': case_payload['case_meta']['case_id'],
        'display_id': case_payload['case_meta']['display_id'],
        'gif_path': str(gif_path),
        'middle_png': str(middle_png),
        'summary_png': str(summary_png),
        'frames_dir': str(frames_dir),
        'num_slices': case_payload['num_slices'],
        'mean_fg_dice': round(case_payload['mean_fg_dice'], 4),
    }


export_rows = []
for dataset_key, row in selected_cases.items():
    try:
        export_rows.append(export_dataset_visuals(dataset_key, row))
    except Exception as exc:
        warnings.warn(f"Failed to export visuals for {dataset_key}: {exc}")

export_summary = pd.DataFrame(export_rows)
display(export_summary)


,dataset,case_id,display_id,gif_path,middle_png,summary_png,frames_dir,num_slices,mean_fg_dice
0,ACDC,patient145_ED,patient145 ED,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,10,0.8324
1,M&M,patient105_ES,patient105 ES,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,12,0.8517
2,STONE,0185_2_0000,0185 seq 2,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,/mnt/hdd/ttoxopeus/basic_UNet/results/qualitat...,11,0.4522


## Final notes

- **Reused files / utilities**: the notebook follows the preprocessing and overlay conventions already used in `src/notebooks/baseline_analysis.ipynb` and loads the UNet implementation from `src/models/unet.py`.
- **Prediction source**: predictions are generated on the fly from the configured checkpoints rather than from pre-saved PNG samples, because the evaluation folders only contain example images rather than full prediction volumes.
- **STONE assumption**: the newer `exp85_stone_l1` baseline metadata appears inconsistent, so the notebook uses `exp54_stone` for STONE qualitative inference.
- **Slice ordering**: slices are shown in increasing stored index along `axis=2`. This ordering was consistent with existing repo utilities, but explicit basal-to-apical verification from metadata was not found.
